# Redact Word documents using the ASE analyzer

Analyze `.docx` or legacy `.doc` through your ASE analyzer, replace detected text with `[REDACTED]`, OCR embedded PNG/JPEG pictures, and paint detected image words black. Build a **new DOCX package**, not a cosmetically masked copy of the original. Body paragraphs, basic bold/italic text, tables, and referenced headers/footers are retained; pagination, fonts, styles, numbering, and advanced layout are not preserved. Legacy DOC is converted with LibreOffice and always produces DOCX output.

Comments, document properties, custom XML, original hyperlink targets, unused package parts, image metadata, and directly hidden runs are not copied. Unsupported content fails rather than producing an apparently safe output. **PII that OCR or the analyzer misses can remain. This is a detected-content redaction workflow, not a privacy certification.**

Copy this notebook **and `ase_document_tools.py`** to the kernel host. No local Presidio engines or anonymizer service are needed: replacements and image edits happen locally after remote detection.

Install Python dependencies on the kernel host:
```powershell
uv pip install --python .venv-ase\Scripts\python.exe requests azure-identity ipykernel Pillow
```
Install **Tesseract plus language data** for embedded pictures and **LibreOffice** for legacy DOC. These are native programs. Use updated tools and trusted inputs; conversion disables macros in an isolated temporary profile but is not a malware sandbox. Python 3.10 through 3.14 is supported by the repository.

Use synthetic test documents first. Do not print source text, tokens, raw HTTP errors, or whole outputs. Keep original and output documents protected and clear outputs before sharing.


In [ ]:
import os
import sys
from pathlib import Path

TOOLS_DIRECTORY = Path(os.environ.get("ASE_SAMPLE_DIRECTORY", str(Path.cwd())))
ANALYZER_BASE_URL = os.environ.get("ANALYZER_BASE_URL", "https://presidio-analyzer.ase-jtl.appserviceenvironment.us")
ANALYZER_API_RESOURCE = os.environ.get("ANALYZER_API_RESOURCE", "")
MANAGED_IDENTITY_CLIENT_ID = os.environ.get("MANAGED_IDENTITY_CLIENT_ID", "")
INPUT_DOCUMENT = Path(os.environ.get("INPUT_DOCUMENT", str(Path("sample_data") / "sample.docx")))
OUTPUT_DOCUMENT = Path(os.environ.get("OUTPUT_DOCUMENT", str(INPUT_DOCUMENT.with_name(INPUT_DOCUMENT.stem + "_redacted.docx"))))
LANGUAGE = "en"
OCR_LANGUAGE = "eng"
OCR_MIN_CONFIDENCE = 50
TESSERACT_COMMAND = os.environ.get("TESSERACT_COMMAND", "tesseract")
LIBREOFFICE_COMMAND = os.environ.get("LIBREOFFICE_COMMAND", "soffice")


Use your working analyzer Application ID URI **without `/.default`** and VM UMI **client ID**, not principal ID. Retain the Government v1 issuer configuration `https://login.microsoftonline.us/<tenant-id>/` without `/v2.0`, the tenant restriction, and caller allowlists. Keep TLS verification enabled.

Set `TOOLS_DIRECTORY` to the helper folder, and use absolute input/output paths if needed. Outputs must be new files in an existing directory. Set full Tesseract/LibreOffice executable paths if not on PATH. `OCR_LANGUAGE` uses Tesseract codes (such as `eng`), not analyzer ISO codes (such as `en`).


In [ ]:
if not (TOOLS_DIRECTORY / "ase_document_tools.py").is_file():
    raise FileNotFoundError("Copy ase_document_tools.py to the kernel host and set TOOLS_DIRECTORY to its folder.")
sys.path.insert(0, str(TOOLS_DIRECTORY.resolve()))
from ase_document_tools import ASEAnalyzer, redact_word

with ASEAnalyzer(ANALYZER_BASE_URL, ANALYZER_API_RESOURCE, MANAGED_IDENTITY_CLIENT_ID, LANGUAGE) as analyzer:
    summary = redact_word(
        INPUT_DOCUMENT, OUTPUT_DOCUMENT, analyzer.analyze,
        libreoffice=LIBREOFFICE_COMMAND, ocr_language=OCR_LANGUAGE,
        tesseract=TESSERACT_COMMAND, min_confidence=OCR_MIN_CONFIDENCE,
    )
print(f"Saved rebuilt DOCX: {summary['text_detections']} text detections and {summary['image_detections']} image detections. Review for missed PII.")


## Fail-closed boundaries and review

Rejects multiple sections, footnotes/endnotes, tracked changes, fields (including page numbers and TOCs), text boxes, content controls, equations, floating/vector/multi-frame drawings, charts/SmartArt, embedded OLE objects, macros, linked external content other than hyperlinks, and style-based hidden text. Directly hidden runs are dropped. Unsupported body/run content is not silently copied. Split or simplify a copy before processing.

Low-confidence OCR or no OCR text on a nonuniform picture refuses the output; no-image DOCX files do not need OCR execution. Basic text/table structure is retained, not original advanced formatting. Images use black boxes; `[REDACTED]` labels apply to Word text. Overlapping detections are merged, including matches crossing run boundaries.

Review output privately. Also validate extraction and image pixels against synthetic ground truth, not just the notebook's success message. Missing detections cannot be fixed by masking or metadata cleanup. No output is published on processing failure; existing files are never replaced. The final filesystem must support same-directory hard links for atomic publication.
